# Evaluate Term Dispersion Scores on the GENIA Corpus Data and Reproduce Results Reported in the Mannuscript 

Authors: Samuel Sarria Hurtado, Uyen "Rachel" Lai, and Paul Sheridan

Description: Evaluate the following term dispersion scores on the Genia corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)

Calculate average P@k scores for each scoring function using the GENIA terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords. 

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the GENIA Corpus Data

In particular, we load the preprocessed GENIA corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., amino_acid, nucleotide, multi_cell, cell, and other).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed GENIA corpus documents
genia_corpus_path = '../1-preprocessing/GENIAcorpus3.02-preprocessed.json'

with open(genia_corpus_path, "r") as j:
  genia_corpus = json.loads(j.read())

# Load gold standard terms 
genia_keywords_path = '../1-preprocessing/GENIAcorpus3.02-keywords.tsv'

with open(genia_keywords_path, "r") as c:
  genia_lexical_units_and_sems = pd.read_csv(c, sep='\t')

genia_lexical_units = genia_lexical_units_and_sems.lex.to_numpy()

# Hardcode the low-level semantic classes and their associated high-level abstract semantic classes
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

Process high-level semantic classes.

In [3]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Create data frame of lexical units, semantic classes, and high-level semantic classes
vocab_size = len(genia_lexical_units_and_sems) # Number of terms in the vocabulary
high_level_sems_lst = [] # Initialize list for recording high-level semantic classes

# For each term in the vocab, identify low-level semantic class with high-level one 
for index in range(vocab_size):
    low_level_sem = genia_lexical_units_and_sems.iloc[index, 1]
    if low_level_sem in amino_acid_sems:
        high_level_sems_lst.append('amino_acid')
    elif low_level_sem in nucleotide_sems:
        high_level_sems_lst.append('nucleotide')
    elif low_level_sem in multi_cell_sems:
        high_level_sems_lst.append('multi_cell')
    elif low_level_sem in cell_sems:
        high_level_sems_lst.append('cell')
    else:
        high_level_sems_lst.append('other')

# Add high-level semantic classes to data frame
genia_lexical_units_and_sems['class'] = high_level_sems_lst

# Print to console:
display(genia_lexical_units_and_sems)

,lex,sem,class
0,IL-2_gene_expression_lex,G#other_name,other
1,IL-2_gene_lex,G#DNA_domain_or_region,nucleotide
2,NF-kappa_B_activation_lex,G#other_name,other
3,NF-kappa_B_lex,G#protein_molecule,amino_acid
4,CD28_lex,G#protein_molecule,amino_acid
...,...,...,...
31782,gp160-induced_AP-1_complex_lex,G#protein_complex,amino_acid
31783,protein_synthesis-independent_lex,G#other_name,other
31784,calcium_channel_blocker_lex,G#other_organic_compound,amino_acid
31785,anti-CD3-induced_interleukin-2_secretion_lex,G#other_name,other


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [5]:
# Compile the GENIA corpus vocabulary
pre_vocab = []
for i in range(len(genia_corpus)):
  pre_vocab.append(genia_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert GENIA documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(genia_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [6]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [7]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/genia/3-tables/../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [26]:
print(counter.get_feature_names_out())

x = pd.DataFrame(data=
                         {'lex': counter.get_feature_names_out(),
                          'IDF': IDF.A[0],
                          'ICF': ICF.A[0],
                          'Chi-sq': Chisq,
                          'CG': CG.A[0],
                          'ICB': ICB.A[0],
                          'DoP': DoP.A[0],
                          'RICF': RICF.A[0],
                          'bi': B_i.A[0],
                          'ni': N_i.A[0]})

merged_df = pd.merge(x, genia_lexical_units_and_sems, on='lex', how='left')

new_order = ["lex", "sem", "class", "bi", "ni", "IDF", "ICF", "Chi-sq", "CG", "ICB", "DoP", "RICF"]
merged_df = merged_df.reindex(columns=new_order)
merged_df = merged_df.rename(columns={'lex': 'term'})
merged_df = merged_df.drop_duplicates()

display(merged_df)

all_duplicates = merged_df[merged_df.duplicated(keep=False)]
#print(all_duplicates)

# Write to TSV
merged_df.to_csv('term-dispersion-scores.tsv', sep='\t')

["'aged'_lymphocyte_lex" "'converted'_TCEd_motif_lex" "'latency_I'_lex"
 ... 'zymography_lex' 'zymosan-treated_cell_lex' 'zymosan_lex']


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251


In [30]:
# Initialize dispersion scores data frame
term_scores_df = pd.DataFrame(data=
                         {'term': counter.get_feature_names_out(),
                          'IDF': IDF.A[0],
                          'ICF': ICF.A[0],
                          'Chi-sq': Chisq,
                          'CG': CG.A[0],
                          'ICB': ICB.A[0],
                          'DoP': DoP.A[0],
                          'RICF': RICF.A[0],
                          'bi': B_i.A[0],
                          'ni': N_i.A[0]})

# Print term dispersion scores to console
display(term_scores_df)

# Write to TSV
term_scores_df.to_csv('term-dispersion-scores.tsv', sep='\t')

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
0,'aged'_lymphocyte_lex,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251,1,1
1,'converted'_TCEd_motif_lex,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251,1,1
2,'latency_I'_lex,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251,1,1
3,'latency_II'_lex,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251,1,1
4,'master_regulator_lex,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251,1,1
...,...,...,...,...,...,...,...,...,...,...
40799,zymogen_plasma_factor_X_lex,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251,1,1
40800,zymogen_plasma_factors_VII_lex,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251,1,1
40801,zymography_lex,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251,1,1
40802,zymosan-treated_cell_lex,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251,1,1


## Compile GENIA Corpus Summary Statistics

This is the result of Table 3 from the manuscript.

In [27]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Hardcode the semantic classes according to their high-level designations.
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Count number of semantic classes in each high-level class
sub_class = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Count number of distinct lexical units in each high-level semantic class
amino_acid = get_high_level_semantic_class_words(amino_acid_sems)
nucleotide = get_high_level_semantic_class_words(nucleotide_sems)
multi_cell = get_high_level_semantic_class_words(multi_cell_sems)
cell = get_high_level_semantic_class_words(cell_sems)
other = get_high_level_semantic_class_words(other_sems)

# Count number of lexical units used as annotations for each high-level semantic class
high_level_class_words = [genia_lexical_units, amino_acid, nucleotide, multi_cell, cell, other]
high_level_class_words_counter = [0, 0, 0, 0, 0, 0]
for i in range(len(high_level_class_words)):
  for j in range(len(vocab)):
    if vocab[j] in high_level_class_words[i]:
      high_level_class_words_counter[i] += N_i.A[0][j]

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Count number of semantic subclasses in each high-level class
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Gather distinct lexical units for each high-level semantic class
amino_acid = get_high_level_semantic_class_words(amino_acid_sems)
nucleotide = get_high_level_semantic_class_words(nucleotide_sems)
multi_cell = get_high_level_semantic_class_words(multi_cell_sems)
cell = get_high_level_semantic_class_words(cell_sems)
other = get_high_level_semantic_class_words(other_sems)

# Count number of lexical units used as annotations for each high-level semantic class
lex_by_semamtic_class_counts = [0, 0, 0, 0, 0, 0]
for i in range(len(high_level_semantic_class_names)):
  for j in range(len(vocab)):
    if vocab[j] in high_level_semantic_class_names[i]:
      lex_by_semamtic_class_counts[i] += N_i.A[0][j]

# Define the data
high_level_semantic_classes = ['all', 'amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
annotations_count_zip = zip(high_level_semantic_classes, high_level_class_words_counter)
annotations_dict = dict(annotations_count_zip)
lex_unit_counts = [len(amino_acid), len(nucleotide), len(multi_cell), len(cell), len(other)]
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]
annotations = [
    annotations_dict['amino_acid'],
    annotations_dict['nucleotide'],
    annotations_dict['multi_cell'],
    annotations_dict['cell'],
    annotations_dict['other']]

# Create a data frame
genia_summary_stats_df = pd.DataFrame({
    'Semantic class': ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other'],
    'Sub-class': subclass_counts,
    'Unique terms': lex_unit_counts,
    "Annotations": annotations
})

In [28]:
# Print GENIA summary statistics
display(genia_summary_stats_df)

# Write to CSV
os.makedirs('table-3', exist_ok=True)
genia_summary_stats_df.to_csv('table-3/semantic-class-stats.csv', index=False)

,Semantic class,Sub-class,Unique terms,Annotations
0,amino_acid,15,10155,42478
1,nucleotide,12,5574,11619
2,multi_cell,5,1444,5247
3,cell,4,4051,11626
4,other,1,10560,19999


## Terminology Extraction Task Experiment

Here we reproduce the result of Table 4 from the manuscript.

In [31]:
# Filter out singletons
term_scores_df = term_scores_df[term_scores_df['ni'] > 1]
term_scores_df = term_scores_df.reset_index(drop=True)
display(term_scores_df)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
0,(+)-pentazocine_lex,7.600902,12.107806,311.074036,2.0,522.0,-0.000720,0.692896,1,2
1,-120_lex,7.600902,12.107806,311.074036,2.0,360.0,-0.000496,0.692896,1,2
2,-150_bp_lex,7.600902,11.702341,inf,3.0,711.0,-0.000654,1.098361,1,3
3,-201/-184_NXS_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.692896,1,2
4,-201_and_-130_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.692896,1,2
...,...,...,...,...,...,...,...,...,...,...
14149,zinc_finger_region_lex,6.907755,12.107806,0.688948,1.0,200.5,-0.001106,-0.000532,2,2
14150,zinc_finger_transcription_factor_lex,5.991465,10.855043,122.211280,1.4,201.4,-0.002168,0.335116,5,7
14151,zinc_lex,6.907755,10.721512,inf,4.0,447.0,-0.000576,1.385763,2,8
14152,zone,6.907755,12.107806,0.688948,1.0,334.5,-0.001845,-0.000532,2,2


In [32]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))
  #rbo_dct = dict.fromkeys(measures, [])

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

In [34]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Set number of replicates
R = 5 # For testing purposes; change back to 100 for final results
#R = 100

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
categories = {
    'all': genia_lexical_units,
    'amino_acid': amino_acid,
    'nucleotide': nucleotide,
    'multi_cell': multi_cell,
    'cell': cell,
    'other': other}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                | 0/5 [00:00<?, ?it/s]

r = 0


 20%|█████████████████▌                                                                      | 1/5 [00:58<03:53, 58.47s/it]

r = 1


 40%|███████████████████████████████████▏                                                    | 2/5 [01:56<02:55, 58.43s/it]

...

 60%|████████████████████████████████████████████████████▊                                   | 3/5 [02:55<01:56, 58.46s/it]

r = 3


 80%|██████████████████████████████████████████████████████████████████████▍                 | 4/5 [03:53<00:58, 58.48s/it]

r = 4


100%|████████████████████████████████████████████████████████████████████████████████████████| 5/5 [04:52<00:00, 58.47s/it]


Save evaluation metrics as Pkl files.

In [35]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [36]:
# Calculate mean P@k scores and write to CSV
all_pk_scores_means = calc_score_means(all_pk_scores)
os.makedirs('table-4', exist_ok=True)
pd.DataFrame(all_pk_scores_means[0]).to_csv('table-4/all-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[1]).to_csv('table-4/amino_acid-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[2]).to_csv('table-4/nucleotide-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[3]).to_csv('table-4/multicell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[4]).to_csv('table-4/cell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[5]).to_csv('table-4/other-pk-means.csv', index=False)

# Calculate standard deviations for P@k scores and write to CSV
all_pk_scores_sds = calc_score_sds(all_pk_scores)
os.makedirs('table-11', exist_ok=True)
pd.DataFrame(all_pk_scores_sds[0]).to_csv('table-11/all-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[1]).to_csv('table-11/amino_acid-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[2]).to_csv('table-11/nucleotide-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[3]).to_csv('table-11/multicell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[4]).to_csv('table-11/cell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[5]).to_csv('table-11/other-pk-sds.csv', index=False)

In [37]:
# Display mean P@k scores and console
print("Mean P@k scores:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_means[0].round(4))
    display(all_pk_scores_means[1].round(4))
    display(all_pk_scores_means[2].round(4))
    display(all_pk_scores_means[3].round(4))
    display(all_pk_scores_means[4].round(4))
    display(all_pk_scores_means[5].round(4))

Mean P@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.8600,0.6200,0.940,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000
50,0.9120,0.6840,0.956,0.9600,1.0000,0.9600,1.0000,0.0200,0.1200
100,0.9260,0.7220,0.942,0.9800,0.9800,0.9400,1.0000,0.0300,0.1100
500,0.9244,0.7652,0.950,0.9832,0.9740,0.9560,0.9920,0.0920,0.1940
1000,0.9268,0.7680,0.951,0.9812,0.9632,0.9512,0.9850,0.1476,0.2494
5000,0.8796,0.7681,0.915,0.9277,0.8992,0.8882,0.9314,0.3833,0.4804


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.2600,0.2000,0.5800,1.0000,0.8000,0.2000,1.0000,0.0000,0.0000
50,0.3760,0.2160,0.5520,0.7560,0.7000,0.4000,0.7920,0.0200,0.0800
100,0.3900,0.2560,0.5440,0.8200,0.7460,0.4600,0.8300,0.0200,0.0800
500,0.3836,0.2440,0.5372,0.6876,0.6220,0.4332,0.6900,0.0500,0.1180
1000,0.3850,0.2486,0.5394,0.6488,0.5900,0.4180,0.6448,0.0738,0.1544
5000,0.3577,0.2516,0.4295,0.4284,0.4144,0.3614,0.4296,0.1660,0.2479


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.2200,0.1200,0.1400,0.0000,0.1000,0.2000,0.0000,0.0000,0.0000
50,0.1720,0.1240,0.1680,0.1000,0.1200,0.1200,0.1000,0.0000,0.0000
100,0.1760,0.1340,0.1540,0.1000,0.1000,0.1300,0.1100,0.0000,0.0000
500,0.1776,0.1508,0.1536,0.1400,0.1460,0.1348,0.1440,0.0040,0.0160
1000,0.1772,0.1408,0.1568,0.1320,0.1396,0.1580,0.1342,0.0140,0.0150
5000,0.1560,0.1328,0.1524,0.1558,0.1515,0.1552,0.1558,0.0517,0.0665


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0200,0.0000,0.0400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0360,0.0280,0.0400,0.0400,0.0400,0.0200,0.0400,0.0000,0.0200
100,0.0420,0.0360,0.0420,0.0200,0.0200,0.0200,0.0200,0.0000,0.0200
500,0.0472,0.0408,0.0424,0.0336,0.0320,0.0660,0.0324,0.0080,0.0120
1000,0.0434,0.0412,0.0414,0.0360,0.0422,0.0580,0.0360,0.0110,0.0150
5000,0.0434,0.0410,0.0437,0.0434,0.0410,0.0454,0.0440,0.0203,0.0225


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.1000,0.1800,0.0200,0.0000,0.0000,0.1000,0.0000,0.0000,0.0000
50,0.1080,0.0880,0.0520,0.0040,0.0000,0.0800,0.0080,0.0000,0.0200
100,0.1000,0.0800,0.0640,0.0100,0.0200,0.0600,0.0100,0.0100,0.0100
500,0.0976,0.0948,0.0752,0.0408,0.0600,0.0880,0.0420,0.0200,0.0300
1000,0.1026,0.1012,0.0782,0.0580,0.0688,0.0910,0.0598,0.0254,0.0376
5000,0.1009,0.0949,0.0954,0.0996,0.1010,0.0994,0.1008,0.0509,0.0526


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.2600,0.1200,0.1600,0.0000,0.1000,0.5000,0.0000,0.0000,0.0000
50,0.2200,0.2280,0.1440,0.0600,0.1400,0.3400,0.0600,0.0000,0.0000
100,0.2180,0.2160,0.1380,0.0300,0.0940,0.2700,0.0300,0.0000,0.0000
500,0.2184,0.2348,0.1416,0.0812,0.1140,0.2340,0.0836,0.0100,0.0180
1000,0.2186,0.2362,0.1352,0.1064,0.1226,0.2262,0.1102,0.0234,0.0274
5000,0.2214,0.2478,0.1940,0.2006,0.1912,0.2268,0.2014,0.0944,0.0908


In [38]:
# Displaye standard deviation of P@k scores to console
print("P@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_sds[0].round(4))
    display(all_pk_scores_sds[1].round(4))
    display(all_pk_scores_sds[2].round(4))
    display(all_pk_scores_sds[3].round(4))
    display(all_pk_scores_sds[4].round(4))
    display(all_pk_scores_sds[5].round(4))

P@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0894,0.1924,0.0894,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0415,0.0410,0.0261,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0182,0.0377,0.0217,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0048,0.0202,0.0087,0.0011,0.0000,0.0000,0.0000,0.0000,0.0014
1000,0.0036,0.0076,0.0055,0.0019,0.0008,0.0004,0.0010,0.0009,0.0005
5000,0.0015,0.0004,0.0007,0.0006,0.0000,0.0000,0.0008,0.0005,0.0009


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.1673,0.2000,0.1304,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0921,0.0537,0.0912,0.0089,0.0000,0.0000,0.0110,0.0000,0.0000
100,0.0308,0.0270,0.0635,0.0000,0.0055,0.0000,0.0000,0.0000,0.0000
500,0.0177,0.0118,0.0190,0.0026,0.0000,0.0011,0.0040,0.0000,0.0014
1000,0.0127,0.0088,0.0140,0.0043,0.0007,0.0000,0.0022,0.0013,0.0011
5000,0.0015,0.0004,0.0011,0.0010,0.0000,0.0000,0.0005,0.0003,0.0013


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.1789,0.1095,0.0548,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0363,0.0654,0.0576,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0251,0.0344,0.0241,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0062,0.0132,0.0094,0.0014,0.0000,0.0011,0.0037,0.0000,0.0000
1000,0.0049,0.0070,0.0098,0.0022,0.0005,0.0000,0.0040,0.0000,0.0000
5000,0.0018,0.0002,0.0011,0.0003,0.0001,0.0000,0.0005,0.0001,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0447,0.0000,0.0548,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0219,0.0228,0.0469,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
100,0.0179,0.0182,0.0415,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0099,0.0054,0.0104,0.0017,0.0000,0.0,0.0009,0.0000,0.0000
1000,0.0077,0.0020,0.0068,0.0032,0.0008,0.0,0.0012,0.0000,0.0000
5000,0.0008,0.0003,0.0006,0.0002,0.0001,0.0,0.0004,0.0001,0.0002


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0707,0.0837,0.0447,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0576,0.0110,0.0228,0.0089,0.0000,0.0,0.0110,0.0000,0.0000
100,0.0292,0.0071,0.0182,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0124,0.0112,0.0147,0.0033,0.0000,0.0,0.0020,0.0000,0.0000
1000,0.0069,0.0107,0.0123,0.0019,0.0011,0.0,0.0022,0.0005,0.0005
5000,0.0010,0.0002,0.0010,0.0012,0.0001,0.0,0.0003,0.0001,0.0006


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.2702,0.0837,0.1342,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0632,0.0415,0.0219,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0295,0.0251,0.0164,0.0000,0.0055,0.0000,0.0000,0.0000,0.0000
500,0.0134,0.0170,0.0143,0.0027,0.0000,0.0000,0.0022,0.0000,0.0000
1000,0.0090,0.0058,0.0033,0.0035,0.0005,0.0004,0.0019,0.0005,0.0005
5000,0.0014,0.0002,0.0006,0.0019,0.0001,0.0000,0.0006,0.0001,0.0015


In [39]:
# Calculate mean R@k scores and write to CSV
all_rk_scores_means = calc_score_means(all_rk_scores)
os.makedirs('table-5', exist_ok=True)
pd.DataFrame(all_rk_scores_means[0]).to_csv('table-5/all-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[1]).to_csv('table-5/amino_acid-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[2]).to_csv('table-5/nucleotide-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[3]).to_csv('table-5/multicell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[4]).to_csv('table-5/cell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[5]).to_csv('table-5/other-rk-means.csv', index=False)

# Calculate standard deviations for R@k scores and write to CSV
all_rk_scores_sds = calc_score_sds(all_rk_scores)
os.makedirs('table-12', exist_ok=True)
pd.DataFrame(all_rk_scores_sds[0]).to_csv('table-12/all-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[1]).to_csv('table-12/amino_acid-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[2]).to_csv('table-12/nucleotide-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[3]).to_csv('table-12/multicell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[4]).to_csv('table-12/cell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[5]).to_csv('table-12/other-rk-sds.csv', index=False)

In [40]:
# Display mean R@k scores console
print("Mean R@k scores:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_means[0].round(4))
    display(all_rk_scores_means[1].round(4))
    display(all_rk_scores_means[2].round(4))
    display(all_rk_scores_means[3].round(4))
    display(all_rk_scores_means[4].round(4))
    display(all_rk_scores_means[5].round(4))

Mean R@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0003,0.0002,0.0003,0.0003,0.0003,0.0003,0.0003,0.0000,0.0000
50,0.0014,0.0011,0.0015,0.0015,0.0016,0.0015,0.0016,0.0000,0.0002
100,0.0029,0.0023,0.0030,0.0031,0.0031,0.0030,0.0031,0.0001,0.0003
500,0.0145,0.0120,0.0149,0.0155,0.0153,0.0150,0.0156,0.0014,0.0031
1000,0.0292,0.0242,0.0299,0.0309,0.0303,0.0299,0.0310,0.0046,0.0078
5000,0.1384,0.1208,0.1439,0.1459,0.1414,0.1397,0.1465,0.0603,0.0756


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0003,0.0002,0.0006,0.0010,0.0008,0.0002,0.0010,0.0000,0.0000
50,0.0019,0.0011,0.0027,0.0037,0.0034,0.0020,0.0039,0.0001,0.0004
100,0.0038,0.0025,0.0054,0.0081,0.0073,0.0045,0.0082,0.0002,0.0008
500,0.0189,0.0120,0.0265,0.0339,0.0306,0.0213,0.0340,0.0025,0.0058
1000,0.0379,0.0245,0.0531,0.0639,0.0581,0.0412,0.0635,0.0073,0.0152
5000,0.1761,0.1239,0.2115,0.2109,0.2040,0.1779,0.2115,0.0818,0.1221


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0004,0.0002,0.0003,0.0000,0.0002,0.0004,0.0000,0.0000,0.0000
50,0.0015,0.0011,0.0015,0.0009,0.0011,0.0011,0.0009,0.0000,0.0000
100,0.0032,0.0024,0.0028,0.0018,0.0018,0.0023,0.0020,0.0000,0.0000
500,0.0159,0.0135,0.0138,0.0126,0.0131,0.0121,0.0129,0.0004,0.0014
1000,0.0318,0.0253,0.0281,0.0237,0.0250,0.0283,0.0241,0.0025,0.0027
5000,0.1400,0.1192,0.1367,0.1398,0.1359,0.1392,0.1398,0.0464,0.0597


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0001,0.0000,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0012,0.0010,0.0014,0.0014,0.0014,0.0007,0.0014,0.0000,0.0007
100,0.0029,0.0025,0.0029,0.0014,0.0014,0.0014,0.0014,0.0000,0.0014
500,0.0163,0.0141,0.0147,0.0116,0.0111,0.0229,0.0112,0.0028,0.0042
1000,0.0301,0.0285,0.0287,0.0249,0.0292,0.0402,0.0249,0.0076,0.0104
5000,0.1504,0.1418,0.1512,0.1503,0.1421,0.1572,0.1522,0.0702,0.0778


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0002,0.0004,0.0000,0.0000,0.0000,0.0002,0.0000,0.0000,0.0000
50,0.0013,0.0011,0.0006,0.0000,0.0000,0.0010,0.0001,0.0000,0.0002
100,0.0025,0.0020,0.0016,0.0002,0.0005,0.0015,0.0002,0.0002,0.0002
500,0.0120,0.0117,0.0093,0.0050,0.0074,0.0109,0.0052,0.0025,0.0037
1000,0.0253,0.0250,0.0193,0.0143,0.0170,0.0225,0.0148,0.0063,0.0093
5000,0.1246,0.1171,0.1177,0.1229,0.1247,0.1227,0.1244,0.0628,0.0650


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0002,0.0001,0.0002,0.0000,0.0001,0.0005,0.0000,0.0000,0.0000
50,0.0010,0.0011,0.0007,0.0003,0.0007,0.0016,0.0003,0.0000,0.0000
100,0.0021,0.0020,0.0013,0.0003,0.0009,0.0026,0.0003,0.0000,0.0000
500,0.0103,0.0111,0.0067,0.0038,0.0054,0.0111,0.0040,0.0005,0.0009
1000,0.0207,0.0224,0.0128,0.0101,0.0116,0.0214,0.0104,0.0022,0.0026
5000,0.1048,0.1173,0.0919,0.0950,0.0905,0.1074,0.0953,0.0447,0.0430


In [41]:
# Display standard deviation of R@k scores to console
print("R@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_sds[0].round(4))
    display(all_rk_scores_sds[1].round(4))
    display(all_rk_scores_sds[2].round(4))
    display(all_rk_scores_sds[3].round(4))
    display(all_rk_scores_sds[4].round(4))
    display(all_rk_scores_sds[5].round(4))

R@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0000,0.0001,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0000
50,0.0001,0.0001,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0000
100,0.0001,0.0001,0.0001,0.0000,0.0,0.0,0.0000,0.0000,0.0000
500,0.0001,0.0003,0.0001,0.0000,0.0,0.0,0.0000,0.0000,0.0000
1000,0.0001,0.0002,0.0002,0.0001,0.0,0.0,0.0000,0.0000,0.0000
5000,0.0002,0.0001,0.0001,0.0001,0.0,0.0,0.0001,0.0001,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0002,0.0002,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0005,0.0003,0.0004,0.0000,0.0000,0.0000,0.0001,0.0000,0.0000
100,0.0003,0.0003,0.0006,0.0000,0.0001,0.0000,0.0000,0.0000,0.0000
500,0.0009,0.0006,0.0009,0.0001,0.0000,0.0001,0.0002,0.0000,0.0001
1000,0.0013,0.0009,0.0014,0.0004,0.0001,0.0000,0.0002,0.0001,0.0001
5000,0.0008,0.0002,0.0006,0.0005,0.0000,0.0000,0.0002,0.0001,0.0006


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0003,0.0002,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0003,0.0006,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0005,0.0006,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0006,0.0012,0.0008,0.0001,0.0000,0.0001,0.0003,0.0000,0.0000
1000,0.0009,0.0013,0.0018,0.0004,0.0001,0.0000,0.0007,0.0000,0.0000
5000,0.0016,0.0002,0.0010,0.0003,0.0001,0.0000,0.0005,0.0001,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0003,0.0000,0.0004,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0008,0.0008,0.0016,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
100,0.0012,0.0013,0.0029,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0034,0.0019,0.0036,0.0006,0.0000,0.0,0.0003,0.0000,0.0000
1000,0.0053,0.0014,0.0047,0.0022,0.0006,0.0,0.0008,0.0000,0.0000
5000,0.0029,0.0009,0.0019,0.0008,0.0003,0.0,0.0015,0.0004,0.0008


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0002,0.0002,0.0001,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0007,0.0001,0.0003,0.0001,0.0000,0.0,0.0001,0.0000,0.0000
100,0.0007,0.0002,0.0004,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0015,0.0014,0.0018,0.0004,0.0000,0.0,0.0002,0.0000,0.0000
1000,0.0017,0.0026,0.0030,0.0005,0.0003,0.0,0.0005,0.0001,0.0001
5000,0.0012,0.0002,0.0012,0.0015,0.0001,0.0,0.0003,0.0001,0.0007


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF,bi,ni
10,0.0003,0.0001,0.0001,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0003,0.0002,0.0001,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
100,0.0003,0.0002,0.0002,0.0000,0.0001,0.0,0.0000,0.0000,0.0000
500,0.0006,0.0008,0.0007,0.0001,0.0000,0.0,0.0001,0.0000,0.0000
1000,0.0009,0.0006,0.0003,0.0003,0.0001,0.0,0.0002,0.0001,0.0001
5000,0.0007,0.0001,0.0003,0.0009,0.0000,0.0,0.0003,0.0001,0.0007


In [ ]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-6', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-6/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-6/amino_acid-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-6/nucleotide-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[3]).to_csv('table-6/multicell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[4]).to_csv('table-6/cell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[5]).to_csv('table-6/other-fk-means.csv', index=False)

# Calculate standard deviations for F1@k scores and write to CSV
all_fk_scores_sds = calc_score_sds(all_fk_scores)
os.makedirs('table-13', exist_ok=True)
pd.DataFrame(all_fk_scores_sds[0]).to_csv('table-13/all-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[1]).to_csv('table-13/amino_acid-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[2]).to_csv('table-13/nucleotide-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[3]).to_csv('table-13/multicell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[4]).to_csv('table-13/cell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[5]).to_csv('table-13/other-fk-sds.csv', index=False)

In [ ]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))
    display(all_fk_scores_means[3].round(4))
    display(all_fk_scores_means[4].round(4))
    display(all_fk_scores_means[5].round(4))

In [ ]:
# Display standard deviation of F1@k scores to console
print("F1@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_sds[0].round(4))
    display(all_fk_scores_sds[1].round(4))
    display(all_fk_scores_sds[2].round(4))
    display(all_fk_scores_sds[3].round(4))
    display(all_fk_scores_sds[4].round(4))
    display(all_fk_scores_sds[5].round(4))

In [ ]:
# Calculate mean RBO scores
mean_rbo_scores = calc_mean_rbo_scores(all_rbo_scores)

# Write to CSV
os.makedirs('table-7', exist_ok=True)
pd.DataFrame(mean_rbo_scores).to_csv('table-7/rbo-means.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores.round(4))

In [ ]:
# Calculate mean RBO scores by semantic class
mean_rbo_scores2 = calc_mean_rbo_scores(all_rbo_scores2)

# Write to CSV
os.makedirs('table-8', exist_ok=True)
pd.DataFrame(mean_rbo_scores2).to_csv('table-8/rbo-means-by-semantic-class.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores2.round(4))

## Top 10 Ranked Terms Example

Here we reproduce the result of Table 5 from the manuscript.

In [ ]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

In [ ]:
ranked_terms_df = resort(term_scores_df)
top_10_ranked_terms_df = pd.DataFrame(top_k(ranked_terms_df, 10))
display(top_10_ranked_terms_df)

In [ ]:
# Write to CSV
os.makedirs('table-9', exist_ok=True)
top_10_ranked_terms_df.to_csv('table-9/top-10-terms.csv', index=False)

## Stopwords Exploratory Analysis

Here we reproduce the result of Table 6 from the manuscript.

In [ ]:
import pandas as pd
from nltk.corpus import stopwords

# Ensure you have the stopwords downloaded
import nltk
nltk.download('stopwords')

def getrank(sorted_measures):
    unique_terms = set()
    for terms in sorted_measures.values():
        unique_terms.update(terms)
    unique_terms = sorted(unique_terms)
    
    # Create a data frame to hold the rankings
    ranking_df = pd.DataFrame(index=unique_terms, columns=sorted_measures.keys())
    
    # Fill the data frame with rankings
    for measure, terms in sorted_measures.items():
        for rank, term in enumerate(terms):
            ranking_df.at[term, measure] = rank + 1  # Rank starts from 1
    
    # Replace NaN with a large number to indicate unranked terms
    ranking_df = ranking_df.fillna(len(unique_terms) + 1)
    #csv_file_path = 'ranking_table.csv'
    #ranking_df.to_csv(csv_file_path)
    return ranking_df

# Function to filter stopwords from the ranking data frame
def filter_stopwords(ranking_df):
    stopwords_list = set(stopwords.words('english'))
    
    # Filter the data frame to include only stopwords
    stopwords_rank = ranking_df[ranking_df.index.isin(stopwords_list)]
    
    # Save the stopwords ranking data frame to a CSV file
    #csv_file_path = 'stopwords_ranking_table.csv'
    #stopwords_rank.to_csv(csv_file_path)
    
    return stopwords_rank

In [ ]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Generate term dispersion ranks for R different versions of the data
R = 100
all_quantiles_df = []
for r in tqdm(range(R)):
    sorted_measures = resort(term_scores_df)
    rank = getrank(sorted_measures)
    stopwords_ranks_df = filter_stopwords(rank)
    bursty_measure_names = stopwords_ranks_df.head(0)
    quantiles = []
    for bursty_measure_name in bursty_measure_names:
        quantiles.append(stopwords_ranks_df[bursty_measure_name].quantile([0, 0.25, 0.5, 0.75, 1]))
    quantiles_df = pd.DataFrame(quantiles)
    all_quantiles_df.append(quantiles_df)

In [ ]:
# Extract the column and index names from the first quantiles data frame
columns = all_quantiles_df[0].columns
index = all_quantiles_df[0].index

# Initialize empty data frames to store the mean and standard deviation values
mean_df = pd.DataFrame(index=index, columns=columns)
std_df = pd.DataFrame(index=index, columns=columns)

# Compute the mean and standard deviation of corresponding elements across all matrices
for col in columns:
    for idx in index:
        values = [matrix.at[idx, col] for matrix in all_quantiles_df]
        mean_df.at[idx, col] = np.mean(values)
        std_df.at[idx, col] = np.std(values)

In [ ]:
# Display the resulting data frames
print("Mean values:")
with pd.option_context('display.precision', 4):
    display(mean_df)
print("\nStandard deviations:")
with pd.option_context('display.precision', 4):
    display(std_df)

In [ ]:
# Write to CSV
os.makedirs('table-10', exist_ok=True)
mean_df.to_csv('table-10/stopword-rank-means.csv')
std_df.to_csv('table-10/stopword-rank-sds.csv')